# Vidal et al. (2019) Case 2: full-to-full transfer

Case 2 loads a formal Case 1 source checkpoint, transfers only `model_state_dict`, reinitializes the target optimizer/scheduler/scaler, fine-tunes on the target `full_train` split, and evaluates the target `test` split. This notebook defaults to a 2-epoch smoke run for `lg_hg2 -> samsung_30t`; it does not start six 515-epoch formal runs automatically.

Reconstruction assumptions: SOC v2 is a cycler-derived benchmark reference, each record is resampled independently to 1 Hz, `seq_len=50`, `stride=50`, and normalization is fit on target `full_train` only.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, subprocess, sys
import pandas as pd
from IPython.display import Image, display

REPO_ROOT = Path('/content/battery-soc-transfer-benchmark')
DRIVE_ROOT = Path('/content/drive/MyDrive/battery_soc_benchmark')
VIDAL_V2_ROOT = DRIVE_ROOT / 'data/labelled/vidal_sequence_v2'
CASE1_OUTPUT_ROOT = DRIVE_ROOT / 'outputs/vidal_case1'
CASE2_OUTPUT_ROOT = DRIVE_ROOT / 'outputs/vidal_case2'
LOCAL_CACHE_ROOT = Path('/content/battery_soc_cache/vidal_case2')

if not REPO_ROOT.is_dir():
    raise FileNotFoundError(
        f'Missing repo: {REPO_ROOT}. Run notebooks/00_colab_setup.ipynb first, '
        'or clone https://github.com/zhangcastle/battery-soc-transfer-benchmark.git to /content.'
    )
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
print('Paths configured:', VIDAL_V2_ROOT, CASE1_OUTPUT_ROOT, CASE2_OUTPUT_ROOT, sep='
')

## Git, SOC v2, and Case 1 formal checkpoint preflight

In [ ]:
from battery_benchmark.data.vidal_sequence_v2_dataset import validate_case1_roots
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
dirty = bool(subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_ROOT, text=True).strip())
print('Commit:', commit, 'dirty:', dirty)
checks = validate_case1_roots(VIDAL_V2_ROOT)
display(checks)
assert (checks.status == 'PASS').all(), 'Required SOC v2 Drive input is missing'

rows = []
for dataset in ['panasonic_18650pf', 'lg_hg2', 'samsung_30t']:
    root = CASE1_OUTPUT_ROOT / dataset / 'seed_42'
    for meta_path in sorted(root.glob('*/run_metadata.json')):
        run = meta_path.parent
        meta = json.loads(meta_path.read_text())
        ckpt = run / 'checkpoints' / 'epoch_515.pt'
        metrics = run / 'metrics.json'
        if meta.get('dataset') == dataset:
            rows.append({
                'dataset': dataset,
                'run_path': str(run),
                'checkpoint': str(ckpt),
                'checkpoint_exists': ckpt.is_file(),
                'metrics_exists': metrics.is_file(),
                'run_mode': meta.get('run_mode'),
                'development_only': meta.get('development_only'),
                'git_commit': meta.get('git_commit'),
            })
case1_runs = pd.DataFrame(rows)
display(case1_runs)
formal = case1_runs[(case1_runs.run_mode == 'formal') & (case1_runs.development_only == False) & case1_runs.checkpoint_exists & case1_runs.metrics_exists]
print('Formal Case 1 checkpoints found:', len(formal))

## Smoke pair: LG HG2 -> Samsung 30T

This cell requires a formal LG HG2 Case 1 checkpoint. If the table above is empty for `lg_hg2`, run Case 1 formal first or provide `SOURCE_CHECKPOINT` manually.

In [ ]:
SOURCE = 'lg_hg2'
TARGET = 'samsung_30t'
match = formal[formal.dataset == SOURCE]
assert not match.empty, f'Missing formal Case 1 source checkpoint for {SOURCE}'
SOURCE_CHECKPOINT = Path(match.iloc[-1].checkpoint)
print('Using source checkpoint:', SOURCE_CHECKPOINT)

!python {REPO_ROOT}/scripts/run_vidal_case2.py   --source {SOURCE}   --target {TARGET}   --config {REPO_ROOT}/configs/experiments/vidal_case2_smoke.yaml   --input-root {VIDAL_V2_ROOT}   --output-root {CASE2_OUTPUT_ROOT}   --case1-output-root {CASE1_OUTPUT_ROOT}   --source-checkpoint {SOURCE_CHECKPOINT}   --cache-root {LOCAL_CACHE_ROOT}   --run-mode smoke   --device auto   --max-train-windows 32

## Inspect latest smoke result

In [ ]:
pair_root = CASE2_OUTPUT_ROOT / '_smoke' / f'source_{SOURCE}__target_{TARGET}' / 'seed_42'
latest = sorted(pair_root.iterdir(), key=lambda p: p.stat().st_mtime)[-1]
print('Latest Case 2 smoke:', latest)
metadata = json.loads((latest / 'run_metadata.json').read_text())
metrics = json.loads((latest / 'metrics.json').read_text())
comparison = json.loads((latest / 'comparison_to_case1.json').read_text())
print('source checkpoint:', metadata['source_checkpoint']['source_checkpoint_path'])
print('checkpoint sha256:', metadata['source_checkpoint']['source_checkpoint_sha256'])
print('initial_target_mse_before_training:', metadata['initial_target_mse_before_training'])
print('final_target_train_mse:', metadata['final_target_train_mse'])
print('metrics:', metrics)
print('comparison:', comparison)
display(pd.read_csv(latest / 'scaler.json') if False else pd.DataFrame([json.loads((latest / 'scaler.json').read_text())]))
for fig in sorted((latest / 'figures').glob('*.png')):
    display(Image(filename=str(fig)))

## Formal commands

Single formal pair example:

```bash
python scripts/run_vidal_case2.py   --source lg_hg2   --target samsung_30t   --config configs/experiments/vidal_case2.yaml   --input-root /content/drive/MyDrive/battery_soc_benchmark/data/labelled/vidal_sequence_v2   --output-root /content/drive/MyDrive/battery_soc_benchmark/outputs/vidal_case2   --case1-output-root /content/drive/MyDrive/battery_soc_benchmark/outputs/vidal_case1   --source-checkpoint /path/to/lg_hg2/seed_42/<run_id>/checkpoints/epoch_515.pt   --run-mode formal
```

Six-direction run example:

```bash
python scripts/run_vidal_case2.py   --all-pairs   --config configs/experiments/vidal_case2.yaml   --input-root /content/drive/MyDrive/battery_soc_benchmark/data/labelled/vidal_sequence_v2   --output-root /content/drive/MyDrive/battery_soc_benchmark/outputs/vidal_case2   --case1-output-root /content/drive/MyDrive/battery_soc_benchmark/outputs/vidal_case1   --source-checkpoint-root /content/drive/MyDrive/battery_soc_benchmark/outputs/vidal_case1   --run-mode formal
```

The notebook does not launch these formal commands automatically.